# Feature Engineering

## Objective

Load the raw Garmin export through the reusable cleaning pipeline, create transparent running features, and validate each feature before using it in exploratory analysis.

> After changing `src/features.py`, restart the kernel and run this notebook from the top so every result uses the latest function definitions.

## 1. Setup and cleaned data

In [67]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import clean_garmin_activities
from src.features import (
    add_aerobic_efficiency_features,
    add_duration_features,
    add_intensity_features,
    add_recovery_spacing_features,
    add_rolling_distance_features,
    add_speed_features,
    add_speed_quality_features,
    add_terrain_features,
    summarize_weekly_training,
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "run_data.csv"
ATHLETE_MAX_HR = 199
SPEED_DIFFERENCE_THRESHOLD_PCT = 10.0

In [68]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "run_data.csv"

raw_activities = pd.read_csv(DATA_PATH)
runs = clean_garmin_activities(raw_activities)

print("Raw activities:", len(raw_activities))
print("Cleaned running activities:", len(runs))

runs.head()

Raw activities: 169
Cleaned running activities: 169


,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Avg Power,Max Power,Steps,Body Battery Drain,Best Lap Time,Number of Laps,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Running,2026-07-11 09:15:11,False,Toronto Running,19.0,1225,0 days 01:47:07,166,185,4.7,...,259,433,19564,-23,0 days 00:00:01,20,0 days 01:46:49,0 days 01:57:28,111,182
1,Running,2026-07-09 07:40:06,False,Toronto Running,9.47,588,0 days 00:49:47,170,191,4.2,...,264,472,8744,-18,0 days 00:00:19.600000,22,0 days 00:48:59,0 days 01:01:14,75,81
2,Running,2026-07-08 17:36:39,False,Toronto Running,7.51,490,0 days 00:49:20,150,162,3.0,...,223,355,8816,-8,0 days 00:03:16,8,0 days 00:49:13,0 days 00:51:47,91,118
3,Treadmill Running,2026-07-07 09:13:03,False,Treadmill Running,2.61,79,0 days 00:08:11.100000,144,176,2.4,...,386,894,808,-4,0 days 00:08:11.100000,1,0 days 00:07:10,0 days 00:18:08,<NA>,<NA>
4,Treadmill Running,2026-07-06 07:40:51,False,Treadmill Running,4.27,424,0 days 00:46:41,144,155,2.6,...,130,303,4310,-9,0 days 00:00:09.500000,32,0 days 00:32:40,0 days 00:47:16,<NA>,<NA>


In [69]:
runs["Activity Type"].value_counts()
runs.dtypes
runs.isna().sum().sort_values(ascending=False).head(10)

Total Ascent               74
Min Elevation              72
Total Descent              71
Max Elevation              67
Body Battery Drain          3
Avg Vertical Ratio          1
Best Pace                   1
Max Power                   1
Avg Power                   1
Normalized Power® (NP®)     1
dtype: int64

**Result:** The cleaned dataset contains 169 activities: 107 outdoor runs and 62 treadmill runs. Both types are intentionally retained. Missing elevation and the single missing moving-time value remain missing rather than being imputed.

## 2. Duration features

### Test purpose

Confirm that `Moving Time` and `Elapsed Time` convert correctly to numeric minutes and that elapsed time is never shorter than moving time.

In [70]:
featured_runs = add_duration_features(runs)

featured_runs[
    ["Moving Time", "moving_minutes", "Elapsed Time", "elapsed_minutes"]
].head()

,Moving Time,moving_minutes,Elapsed Time,elapsed_minutes
0,0 days 01:46:49,106.816667,0 days 01:57:28,117.466667
1,0 days 00:48:59,48.983333,0 days 01:01:14,61.233333
2,0 days 00:49:13,49.216667,0 days 00:51:47,51.783333
3,0 days 00:07:10,7.166667,0 days 00:18:08,18.133333
4,0 days 00:32:40,32.666667,0 days 00:47:16,47.266667


In [71]:
featured_runs[
    ["moving_minutes", "elapsed_minutes"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
moving_minutes,168.0,46.192589,23.525844,3.150,34.062500,47.000000,57.383333,109.033333
elapsed_minutes,169.0,60.219418,28.644304,4.405,41.283333,57.366667,82.233333,143.783333


In [72]:
(featured_runs["elapsed_minutes"] < featured_runs["moving_minutes"]).sum()

0

**Result:** No activity has elapsed time shorter than moving time. Moving minutes are available for 168 of 169 activities; the unavailable treadmill duration is correctly preserved as missing.

## 3. Average speed

### Test purpose

Confirm that distance and moving time produce numeric kilometres per hour and identify missing or extreme values for review.

In [73]:
featured_runs = add_speed_features(featured_runs)

featured_runs[
    ["Distance", "moving_minutes", "Avg Pace", "average_speed_kmh"]
].head()

,Distance,moving_minutes,Avg Pace,average_speed_kmh
0,19.0,106.816667,5.633333,10.672492
1,9.47,48.983333,5.250000,11.599864
2,7.51,49.216667,6.566667,9.155435
3,2.61,7.166667,3.133333,21.851163
4,4.27,32.666667,10.950000,7.842857


In [74]:
print("Missing average speed:", featured_runs["average_speed_kmh"].isna().sum())
featured_runs[["average_speed_kmh"]].describe().T

Missing average speed: 1


,count,mean,std,min,25%,50%,75%,max
average_speed_kmh,168.0,10.905959,2.377655,6.786451,9.612041,10.67678,11.641409,26.470588


**Result:** Speed is calculated for 168 activities. One value is missing because moving time is unavailable. The unusually high maximum should not be interpreted until the speed-quality comparison is applied.

## 4. Speed-quality comparison

### Test purpose

Compare distance/time-derived speed with Garmin pace-derived speed. Flag differences above 10% without deleting or changing activities.

In [75]:
featured_runs = add_speed_quality_features(
    featured_runs,
    max_difference_pct=SPEED_DIFFERENCE_THRESHOLD_PCT,
)

In [76]:
featured_runs[
    [
        "Date",
        "Activity Type",
        "Distance",
        "Avg Pace",
        "average_speed_kmh",
        "speed_from_pace_kmh",
        "speed_difference_pct",
        "speed_quality_flag",
    ]
].head(10)

,Date,Activity Type,Distance,Avg Pace,average_speed_kmh,speed_from_pace_kmh,speed_difference_pct,speed_quality_flag
0,2026-07-11 09:15:11,Running,19.0,5.633333,10.672492,10.650888,0.202840,False
1,2026-07-09 07:40:06,Running,9.47,5.250000,11.599864,11.428571,1.498809,False
2,2026-07-08 17:36:39,Running,7.51,6.566667,9.155435,9.137056,0.201151,False
3,2026-07-07 09:13:03,Treadmill Running,2.61,3.133333,21.851163,19.148936,14.111628,True
4,2026-07-06 07:40:51,Treadmill Running,4.27,10.950000,7.842857,5.479452,43.132143,True
5,2026-07-05 10:22:05,Running,17.01,5.983333,10.043628,10.027855,0.157290,False
6,2026-07-02 08:06:30,Running,6.93,5.166667,11.663394,11.612903,0.434783,False
7,2026-07-01 09:04:54,Running,13.0,5.550000,10.833333,10.810811,0.208333,False
8,2026-06-30 08:37:43,Treadmill Running,6.15,8.883333,9.413265,6.754221,39.368622,True
9,2026-06-29 09:18:46,Running,8.0,7.100000,8.478069,8.450704,0.323815,False


In [77]:
speed_quality_summary = (
    featured_runs.groupby("Activity Type")["speed_quality_flag"]
    .agg(total_runs="size", comparisons_available="count", flagged_runs="sum")
)
speed_quality_summary["missing_comparison"] = (
    speed_quality_summary["total_runs"]
    - speed_quality_summary["comparisons_available"]
)
speed_quality_summary

,total_runs,comparisons_available,flagged_runs,missing_comparison
Activity Type,,,,
Running,107,107,10,0
Treadmill Running,62,61,37,1


In [78]:
featured_runs[
    [
        "Date",
        "Activity Type",
        "Distance",
        "Moving Time",
        "Avg Pace",
        "average_speed_kmh",
        "speed_from_pace_kmh",
        "speed_difference_pct",
    ]
].sort_values(
    "speed_difference_pct",
    ascending=False,
).head(15)

,Date,Activity Type,Distance,Moving Time,Avg Pace,average_speed_kmh,speed_from_pace_kmh,speed_difference_pct
81,2026-02-10 09:04:52,Treadmill Running,1.66,0 days 00:04:43,11.883333,21.116608,5.049088,318.226148
78,2026-02-17 09:01:58,Treadmill Running,1.81,0 days 00:05:24,11.366667,20.111111,5.278592,280.993827
117,2025-10-15 08:32:30,Treadmill Running,1.12,0 days 00:04:07,12.216667,16.323887,4.911323,232.372470
79,2026-02-16 09:39:02,Treadmill Running,2.74,0 days 00:19:17,23.216667,8.525497,2.584350,229.889369
74,2026-02-24 09:01:07,Treadmill Running,1.74,0 days 00:06:41,12.216667,15.620948,4.911323,218.059850
165,2025-06-20 10:49:02,Treadmill Running,1.75,0 days 00:03:58,6.483333,26.470588,9.254499,186.029412
84,2026-02-03 08:29:08,Treadmill Running,1.34,0 days 00:06:28.400000,11.883333,12.420185,5.049088,145.988671
75,2026-02-21 11:54:32,Treadmill Running,3.67,0 days 00:23:09,14.850000,9.511879,4.040404,135.419006
30,2026-05-20 10:50:55,Running,7.53,0 days 00:35:43,10.416667,12.649557,5.760000,119.610359
71,2026-03-03 09:03:15,Treadmill Running,2.54,0 days 00:09:50,8.150000,15.498305,7.361963,110.518644


**Result:** At the 10% threshold, 10 of 107 outdoor runs and 37 of 62 treadmill runs are flagged; one treadmill activity cannot be compared. Efficiency visuals should separate activity types and apply an explicit quality rule.

## 5. Rolling distance

### Test purpose

Confirm that the 7-day and 28-day windows include the current run, contain no missing totals, and maintain the expected relationship between window sizes.

In [79]:
featured_runs = add_rolling_distance_features(featured_runs)

In [80]:
featured_runs[
    [
        "Date",
        "Distance",
        "rolling_7_day_distance_km",
        "rolling_28_day_distance_km",
    ]
].head(15)

,Date,Distance,rolling_7_day_distance_km,rolling_28_day_distance_km
0,2025-06-14 07:40:11,4.0,4.00,4.00
1,2025-06-17 10:25:09,1.01,5.01,5.01
2,2025-06-19 11:06:18,1.0,6.01,6.01
3,2025-06-20 10:49:02,1.75,7.76,7.76
4,2025-06-22 10:19:23,13.0,16.76,20.76
5,2025-06-25 08:31:40,6.02,21.77,26.78
6,2025-06-26 08:11:36,1.41,23.18,28.19
7,2025-06-26 08:25:17,1.51,24.69,29.70
8,2025-07-01 17:44:10,9.01,17.95,38.71
9,2025-07-06 09:20:13,17.0,26.01,55.71


In [81]:
featured_runs[
    [
        "rolling_7_day_distance_km",
        "rolling_28_day_distance_km",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
rolling_7_day_distance_km,169.0,32.669053,13.904988,4.0,23.85,30.89,42.29,70.87
rolling_28_day_distance_km,169.0,108.696154,47.953931,4.0,74.60,104.36,143.93,206.36


In [82]:
checks = {
    "7-day total below current distance": (
        featured_runs["rolling_7_day_distance_km"]
        < featured_runs["Distance"]
    ).sum(),

    "28-day total below 7-day total": (
        featured_runs["rolling_28_day_distance_km"]
        < featured_runs["rolling_7_day_distance_km"]
    ).sum(),

    "missing 7-day totals": (
        featured_runs["rolling_7_day_distance_km"].isna().sum()
    ),

    "missing 28-day totals": (
        featured_runs["rolling_28_day_distance_km"].isna().sum()
    ),
}

checks

{'7-day total below current distance': 0,
 '28-day total below 7-day total': 0,
 'missing 7-day totals': 0,
 'missing 28-day totals': 0}

**Result:** All validation counts are zero. Every run has both totals, every 7-day total includes at least the current distance, and no 28-day total is below its corresponding 7-day total.

## 6. Weekly training summary

### Test purpose

Confirm that Monday-through-Sunday aggregation preserves total distance, moving time, and activity count.

In [83]:
weekly_training = summarize_weekly_training(featured_runs)
weekly_training.tail(10)

,week_start,weekly_distance_km,weekly_running_minutes,weekly_run_count
45,2026-05-04,60.61,385.233333,6
46,2026-05-11,36.71,152.033333,4
47,2026-05-18,38.27,225.033333,5
48,2026-05-25,49.18,298.800000,8
49,2026-06-01,22.57,134.883333,5
50,2026-06-08,14.01,78.700000,1
51,2026-06-15,15.25,85.683333,2
52,2026-06-22,26.01,154.083333,2
53,2026-06-29,51.09,305.083333,5
54,2026-07-06,42.86,244.850000,5


In [84]:
activity_running_minutes = (
    featured_runs["Moving Time"].dt.total_seconds().sum() / 60
)

weekly_checks = {
    "distance totals match": abs(
        weekly_training["weekly_distance_km"].sum()
        - featured_runs["Distance"].sum()
    ) < 1e-9,
    "activity counts match": (
        weekly_training["weekly_run_count"].sum() == len(featured_runs)
    ),
    "running-time totals match": abs(
        weekly_training["weekly_running_minutes"].sum()
        - activity_running_minutes
    ) < 1e-9,
}

print(weekly_checks)
weekly_training[
    ["weekly_distance_km", "weekly_running_minutes", "weekly_run_count"]
].describe().T

{'distance totals match': True, 'activity counts match': True, 'running-time totals match': True}


,count,mean,std,min,25%,50%,75%,max
weekly_distance_km,55.0,25.192545,13.57786,0.54,14.63,24.65,35.135,60.61
weekly_running_minutes,55.0,141.097364,82.245903,3.15,79.416667,134.883333,180.791667,385.233333
weekly_run_count,55.0,3.072727,1.642655,1.0,2.0,3.0,4.0,8.0


**Result:** All checks pass. The 55 represented weeks preserve 1,385.59 km, 7,760.355 moving minutes, and all 169 activities. Weeks without runs are not inserted.

## 7. Intensity features

### Test purpose

Validate relative heart-rate intensity using the documented 199 bpm maximum and inspect Aerobic TE per moving minute. Both remain exploratory.

In [85]:
featured_runs = add_intensity_features(
    featured_runs,
    athlete_max_hr=ATHLETE_MAX_HR,
)

In [86]:
featured_runs[
    [
        "Avg HR",
        "relative_hr_intensity",
        "Aerobic TE",
        "moving_minutes",
        "aerobic_effect_per_minute",
    ]
].head(10)

,Avg HR,relative_hr_intensity,Aerobic TE,moving_minutes,aerobic_effect_per_minute
0,148,0.743719,3.1,24.966667,0.124166
1,149,0.748744,2.0,6.116667,0.326975
2,172,0.864322,2.0,4.950000,0.404040
3,158,0.793970,2.4,3.966667,0.605042
4,178,0.894472,5.0,66.516667,0.075169
5,171,0.859296,3.9,29.583333,0.131831
6,152,0.763819,2.1,8.381667,0.250547
7,169,0.849246,2.3,7.916667,0.290526
8,170,0.854271,4.3,49.383333,0.087074
9,175,0.879397,5.0,91.033333,0.054925


In [87]:
featured_runs[
    [
        "average_speed_kmh",
        "relative_hr_intensity",
        "aerobic_effect_per_minute",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
average_speed_kmh,168.0,10.905959,2.377655,6.786451,9.612041,10.676780,11.641409,26.470588
relative_hr_intensity,168.0,0.791906,0.064523,0.582915,0.752513,0.798995,0.839196,0.934673
aerobic_effect_per_minute,168.0,0.109208,0.096443,0.035279,0.060983,0.079755,0.105869,0.605042


In [88]:
print("Missing intensity values:")
print(featured_runs[
    ["relative_hr_intensity", "aerobic_effect_per_minute"]
].isna().sum())
print("Relative HR values above 1.0:", (
    featured_runs["relative_hr_intensity"] > 1
).sum())

Missing intensity values:
relative_hr_intensity        1
aerobic_effect_per_minute    1
dtype: int64
Relative HR values above 1.0: 0


**Result:** Both features are available for 168 activities, and no relative-HR value exceeds 1.0. Aerobic TE per minute remains exploratory because Garmin's session-level score may not scale linearly with duration.

## 8. Aerobic-efficiency features

### Test purpose

Validate metres per heartbeat, watts per heartbeat, and speed per watt. Check one row manually and confirm that the two ratio columns are distinct.

In [89]:
featured_runs = add_aerobic_efficiency_features(featured_runs)

In [90]:
featured_runs[
    [
        "average_speed_kmh",
        "Avg HR",
        "Avg Power",
        "metres_per_heartbeat",
        "power_to_hr_ratio",
        "speed_to_power_ratio",
    ]
].head(10)

,average_speed_kmh,Avg HR,Avg Power,metres_per_heartbeat,power_to_hr_ratio,speed_to_power_ratio
0,9.612817,148,278,1.082524,1.878378,0.034578
1,9.907357,149,254,1.108205,1.704698,0.039005
2,12.121212,172,364,1.174536,2.116279,0.0333
3,26.470588,158,168,2.792256,1.063291,0.157563
4,11.726384,178,281,1.097976,1.578652,0.041731
5,12.209577,171,285,1.190017,1.666667,0.042841
6,10.093458,152,151,1.106739,0.993421,0.066844
7,11.444211,169,270,1.12862,1.597633,0.042386
8,10.947013,170,286,1.073237,1.682353,0.038276
9,11.204687,175,277,1.067113,1.582857,0.04045


In [91]:
example = featured_runs.dropna(
    subset=["average_speed_kmh", "Avg HR", "Avg Power"]
).iloc[0]

efficiency_checks = {
    "power-to-HR matches": abs(
        example["power_to_hr_ratio"]
        - example["Avg Power"] / example["Avg HR"]
    ) < 1e-12,
    "speed-to-power matches": abs(
        example["speed_to_power_ratio"]
        - example["average_speed_kmh"] / example["Avg Power"]
    ) < 1e-12,
    "ratio columns are distinct": not featured_runs[
        "power_to_hr_ratio"
    ].equals(featured_runs["speed_to_power_ratio"]),
}

print(efficiency_checks)
featured_runs[
    ["metres_per_heartbeat", "power_to_hr_ratio", "speed_to_power_ratio"]
].describe().T

{'power-to-HR matches': True, 'speed-to-power matches': True, 'ratio columns are distinct': True}


,count,mean,std,min,25%,50%,75%,max
metres_per_heartbeat,168.0,1.154839,0.253442,0.872435,1.054078,1.115306,1.16957,2.792256
power_to_hr_ratio,168.0,1.462501,0.367538,0.378205,1.202807,1.567463,1.670659,2.940741
speed_to_power_ratio,168.0,0.051982,0.026744,0.024087,0.040959,0.042213,0.052301,0.207026


**Result:** All checks pass, and the ratio columns use distinct formulas. Efficiency values exist for 168 activities. Apply the speed-quality rule and separate dissimilar run types before interpretation.

## 9. Terrain features

### Test purpose

Validate climbing metres per kilometre and paused-time ratio. Confirm valid ratio bounds and preserve missing treadmill elevation.

In [92]:
featured_runs = add_terrain_features(featured_runs)

In [93]:
featured_runs[
    [
        "Activity Type",
        "Distance",
        "Total Ascent",
        "climbing_density_m_per_km",
        "moving_minutes",
        "elapsed_minutes",
        "pause_ratio",
    ]
].head(15)

,Activity Type,Distance,Total Ascent,climbing_density_m_per_km,moving_minutes,elapsed_minutes,pause_ratio
0,Running,4.0,9,2.25,24.966667,29.116667,0.142530
1,Running,1.01,12,11.881188,6.116667,6.203333,0.013971
2,Running,1.0,26,26.0,4.950000,5.103333,0.030046
3,Treadmill Running,1.75,<NA>,<NA>,3.966667,11.333333,0.650000
4,Running,13.0,259,19.923077,66.516667,84.016667,0.208292
5,Running,6.02,92,15.282392,29.583333,39.216667,0.245644
6,Running,1.41,12,8.510638,8.381667,10.233333,0.180945
7,Running,1.51,16,10.596026,7.916667,8.866667,0.107143
8,Running,9.01,54,5.993341,49.383333,56.416667,0.124668
9,Running,17.0,107,6.294118,91.033333,112.683333,0.192131


In [94]:
invalid_pause_ratios = (
    (featured_runs["pause_ratio"] < 0)
    | (featured_runs["pause_ratio"] > 1)
).sum()

terrain_availability = (
    featured_runs.groupby("Activity Type")["climbing_density_m_per_km"]
    .agg(total_runs="size", available="count")
)
terrain_availability["missing"] = (
    terrain_availability["total_runs"] - terrain_availability["available"]
)

print("Invalid pause ratios:", invalid_pause_ratios)
print(terrain_availability)
featured_runs[
    ["climbing_density_m_per_km", "pause_ratio"]
].describe().T

Invalid pause ratios: 0
                   total_runs  available  missing
Activity Type                                    
Running                   107         95       12
Treadmill Running          62          0       62


,count,mean,std,min,25%,50%,75%,max
climbing_density_m_per_km,95.0,7.065666,8.138348,0.107643,1.622066,3.421634,9.583511,49.40239
pause_ratio,168.0,0.228854,0.182782,0.000666,0.09036,0.182353,0.316418,0.76098


In [95]:
example = (
    featured_runs[
        featured_runs["Total Ascent"].notna()
    ]
    .iloc[0]
)

print(
    "Expected climbing density:",
    example["Total Ascent"] / example["Distance"],
)

print(
    "Calculated climbing density:",
    example["climbing_density_m_per_km"],
)

print(
    "Expected pause ratio:",
    (
        example["elapsed_minutes"]
        - example["moving_minutes"]
    )
    / example["elapsed_minutes"],
)

print(
    "Calculated pause ratio:",
    example["pause_ratio"],
)

Expected climbing density: 2.25
Calculated climbing density: 2.25
Expected pause ratio: 0.14253005151688616
Calculated pause ratio: 0.14253005151688616


**Result:** No pause ratio is outside zero to one. Climbing density is available for 95 outdoor runs, missing for 12 outdoor runs, and correctly missing for all 62 treadmill runs.

## 10. Recovery spacing

### Test purpose

Confirm nonnegative calendar-day gaps, one missing earliest gap, full row preservation, and plausible longest breaks.

In [96]:
featured_runs = add_recovery_spacing_features(featured_runs)

In [97]:
featured_runs[
    [
        "Date",
        "Activity Type",
        "Distance",
        "days_since_previous_run",
    ]
].head(15)

,Date,Activity Type,Distance,days_since_previous_run
0,2025-06-14 07:40:11,Running,4.0,<NA>
1,2025-06-17 10:25:09,Running,1.01,3
2,2025-06-19 11:06:18,Running,1.0,2
3,2025-06-20 10:49:02,Treadmill Running,1.75,1
4,2025-06-22 10:19:23,Running,13.0,2
5,2025-06-25 08:31:40,Running,6.02,3
6,2025-06-26 08:11:36,Running,1.41,1
7,2025-06-26 08:25:17,Running,1.51,0
8,2025-07-01 17:44:10,Running,9.01,5
9,2025-07-06 09:20:13,Running,17.0,5


In [98]:
recovery_checks = {
    "negative gaps": int((
        featured_runs["days_since_previous_run"] < 0
    ).sum()),
    "missing gaps": int(
        featured_runs["days_since_previous_run"].isna().sum()
    ),
    "rows preserved": len(featured_runs) == len(runs),
    "same-day runs": int((
        featured_runs["days_since_previous_run"] == 0
    ).sum()),
}

print(recovery_checks)
featured_runs[
    ["Date", "Activity Type", "Distance", "days_since_previous_run"]
].sort_values("days_since_previous_run", ascending=False).head(10)

{'negative gaps': 0, 'missing gaps': 1, 'rows preserved': True, 'same-day runs': 10}


,Date,Activity Type,Distance,days_since_previous_run
100,2026-03-20 09:58:54,Treadmill Running,4.6,13
76,2026-01-11 07:47:05,Running,11.72,10
75,2026-01-01 10:40:43,Running,8.09,8
64,2025-11-24 10:39:21,Running,5.52,8
154,2026-06-13 10:07:08,Running,14.01,7
13,2025-07-23 07:10:06,Running,9.5,7
43,2025-09-27 09:08:59,Running,13.02,6
82,2026-01-31 10:17:33,Running,0.54,6
8,2025-07-01 17:44:10,Running,9.01,5
9,2025-07-06 09:20:13,Running,17.0,5


**Result:** There are no negative gaps, exactly one missing value for the earliest activity, and all 169 rows are preserved. Ten activities follow another run on the same date; the longest gap is 13 days.

## 11. Checkpoint and remaining implementation

The reusable layer now covers duration, speed, speed quality, rolling volume, weekly summaries, optional intensity, aerobic-efficiency proxies, terrain, and recovery spacing.

Before declaring feature engineering complete:

1. implement calendar year, month, and ISO week fields;
2. implement the one-call `add_mvp_run_features()` pipeline; and
3. add automated tests for the newer functions.

After those pass, move to concise exploratory visualizations. Keep outdoor and treadmill runs distinct, and apply the speed-quality rule to efficiency analysis.